# Tennis vision — Colab GPU run

> ⚠️ **Generated file** — edit `scripts/build_colab_nb.py`, not this notebook. Manual edits are overwritten on regeneration.

Runs the `yastrebksv_TennisProject` pipeline (ball / court / person / bounce) with the per-axis coordinate-scaling fix applied so overlays align on 1920×1080 input.

**Before running:** `Runtime → Change runtime type → GPU` (A100 on Pro gives best throughput).

Expected: ~1 min end-to-end for `point1.mp4` (475 frames) on A100. Longer for `set1.mp4`.

In [ ]:
!nvidia-smi -L

In [ ]:
!pip install -q scenedetect==0.6.4 catboost gdown ultralytics

In [ ]:
!rm -rf /content/TennisProject
!git clone -q https://github.com/yastrebksv/TennisProject.git /content/TennisProject
!cd /content/TennisProject && git checkout -q b7552e955253c008c7612298cb40694682a74ef9
!sed -i 's/np\.Inf/np.inf/g' /content/TennisProject/homography.py

In [ ]:
%%writefile /content/TennisProject/ball_detector.py
from tracknet import BallTrackerNet
import torch
import cv2
import numpy as np
from scipy.spatial import distance
from tqdm import tqdm

class BallDetector:
    def __init__(self, path_model=None, device='cuda'):
        self.model = BallTrackerNet(input_channels=9, out_channels=256)
        self.device = device
        if path_model:
            self.model.load_state_dict(torch.load(path_model, map_location=device, weights_only=False))
            self.model = self.model.to(device)
            self.model.eval()
        self.width = 640
        self.height = 360

    def infer_model(self, frames):
        """ Run pretrained model on a consecutive list of frames
        :params
            frames: list of consecutive video frames
        :return
            ball_track: list of detected ball points in original frame coordinates
        """
        orig_h, orig_w = frames[0].shape[:2]
        scale_x = orig_w / self.width
        scale_y = orig_h / self.height
        # Preserve original max_dist semantics (tuned for 1280x720 → scale=2).
        max_dist = 80.0 * max(scale_x, scale_y) / 2.0

        ball_track = [(None, None)]*2
        prev_pred = [None, None]
        for num in tqdm(range(2, len(frames))):
            img = cv2.resize(frames[num], (self.width, self.height))
            img_prev = cv2.resize(frames[num-1], (self.width, self.height))
            img_preprev = cv2.resize(frames[num-2], (self.width, self.height))
            imgs = np.concatenate((img, img_prev, img_preprev), axis=2)
            imgs = imgs.astype(np.float32)/255.0
            imgs = np.rollaxis(imgs, 2, 0)
            inp = np.expand_dims(imgs, axis=0)

            # torch.no_grad() prevents per-frame autograd state accumulation;
            # without it 30k sequential forwards can exhaust system RAM.
            with torch.no_grad():
                out = self.model(torch.from_numpy(inp).float().to(self.device))
                output = out.argmax(dim=1).detach().cpu().numpy()
            x_pred, y_pred = self.postprocess(output, prev_pred, scale_x, scale_y, max_dist)
            prev_pred = [x_pred, y_pred]
            ball_track.append((x_pred, y_pred))
        return ball_track

    def postprocess(self, feature_map, prev_pred, scale_x, scale_y, max_dist=80):
        """
        :params
            feature_map: feature map with shape (1, self.height, self.width)
            prev_pred: [x,y] coordinates of ball prediction from previous frame (original-frame space)
            scale_x, scale_y: per-axis scale from network space (self.width, self.height) to original frame
            max_dist: maximum distance (original-frame pixels) from previous ball detection to remove outliers
        :return
            x,y ball coordinates in original frame space
        """
        feature_map *= 255
        feature_map = feature_map.reshape((self.height, self.width))
        feature_map = feature_map.astype(np.uint8)
        ret, heatmap = cv2.threshold(feature_map, 127, 255, cv2.THRESH_BINARY)
        circles = cv2.HoughCircles(heatmap, cv2.HOUGH_GRADIENT, dp=1, minDist=1, param1=50, param2=2, minRadius=2,
                                   maxRadius=7)
        x, y = None, None
        if circles is not None:
            if prev_pred[0]:
                for i in range(len(circles[0])):
                    x_temp = circles[0][i][0]*scale_x
                    y_temp = circles[0][i][1]*scale_y
                    dist = distance.euclidean((x_temp, y_temp), prev_pred)
                    if dist < max_dist:
                        x, y = x_temp, y_temp
                        break
            else:
                x = circles[0][0][0]*scale_x
                y = circles[0][0][1]*scale_y
        return x, y


In [ ]:
%%writefile /content/TennisProject/court_detection_net.py
import cv2
import numpy as np
import torch
from tracknet import BallTrackerNet
import torch.nn.functional as F
from tqdm import tqdm
from postprocess import refine_kps
from homography import get_trans_matrix, refer_kps


OUTPUT_WIDTH = 640
OUTPUT_HEIGHT = 360


class CourtDetectorNet():
    def __init__(self, path_model=None,  device='cuda'):
        self.model = BallTrackerNet(out_channels=15)
        self.device = device
        if path_model:
            self.model.load_state_dict(torch.load(path_model, map_location=device, weights_only=False))
            self.model = self.model.to(device)
            self.model.eval()

    def _infer_single_frame(self, image, scale_x, scale_y):
        """Run one forward pass + keypoint refinement + H fit.

        Returns (kps_projected_or_None, inv_matrix_or_None).
        """
        img = cv2.resize(image, (OUTPUT_WIDTH, OUTPUT_HEIGHT))
        inp = (img.astype(np.float32) / 255.)
        inp = torch.tensor(np.rollaxis(inp, 2, 0))
        inp = inp.unsqueeze(0)

        # torch.no_grad() prevents autograd from retaining per-frame activation
        # graphs. Without it, sequential forward passes accumulate state that
        # balloons system RAM (hit 177 GB on a 30k-frame run).
        with torch.no_grad():
            out = self.model(inp.float().to(self.device))[0]
            pred = F.sigmoid(out).detach().cpu().numpy()

        points = []
        for kps_num in range(14):
            heatmap = (pred[kps_num]*255).astype(np.uint8)
            ret, heatmap = cv2.threshold(heatmap, 170, 255, cv2.THRESH_BINARY)
            circles = cv2.HoughCircles(heatmap, cv2.HOUGH_GRADIENT, dp=1, minDist=20, param1=50, param2=2,
                                       minRadius=10, maxRadius=25)
            if circles is not None:
                x_pred = circles[0][0][0]*scale_x
                y_pred = circles[0][0][1]*scale_y
                if kps_num not in [8, 12, 9]:
                    x_pred, y_pred = refine_kps(image, int(y_pred), int(x_pred), crop_size=40)
                points.append((x_pred, y_pred))
            else:
                points.append(None)

        matrix_trans = get_trans_matrix(points)
        if matrix_trans is None:
            return None, None
        kps = cv2.perspectiveTransform(refer_kps, matrix_trans)
        inv_mat = cv2.invert(matrix_trans)[1]
        return kps, inv_mat

    def infer_model(self, frames):
        """Dense per-frame inference. Retained for parity / diagnostic use."""
        orig_h, orig_w = frames[0].shape[:2]
        scale_x = orig_w / OUTPUT_WIDTH
        scale_y = orig_h / OUTPUT_HEIGHT

        kps_res = []
        matrixes_res = []
        for image in tqdm(frames):
            kps, inv_mat = self._infer_single_frame(image, scale_x, scale_y)
            kps_res.append(kps)
            matrixes_res.append(inv_mat)
        return matrixes_res, kps_res

    def infer_model_sparse(self, frames, scenes, fps, jitter_threshold_px=15.0,
                           samples_per_second=1.0, min_samples_per_scene=3):
        """Scene-aware sparse court detection.

        Broadcast tennis holds a near-static main camera, so running the court
        net on every frame is mostly wasted work — `stabilize_static_camera`
        in main.py was already collapsing per-frame homographies to a single
        median. This method skips directly to sparse sampling.

        For each scene [s, e):
          - Sample ~samples_per_second frames evenly (min min_samples_per_scene).
          - Run the net only on sampled frames.
          - If ≥ min_samples valid: check keypoint MAD jitter across samples.
              * Jitter < threshold → static camera → one median H for all frames.
              * Jitter ≥ threshold → moving camera → nearest-sample H for each frame.
          - If < min_samples valid (replays / tight / crowd): all frames stay None.
            Downstream already handles None homography as "skip this frame".

        Returns (homography_matrices, kps_court) with length == len(frames),
        matching the dense infer_model API exactly.
        """
        total_frames = len(frames)
        matrixes_res = [None] * total_frames
        kps_res = [None] * total_frames

        orig_h, orig_w = frames[0].shape[:2]
        scale_x = orig_w / OUTPUT_WIDTH
        scale_y = orig_h / OUTPUT_HEIGHT

        sample_step = max(1, int(round(fps / samples_per_second)))

        # Build per-scene sample index lists (anchor both endpoints).
        scene_samples = []
        all_sample_idxs = []
        for (s, e) in scenes:
            idxs = list(range(s, e, sample_step))
            if not idxs or idxs[-1] != e - 1:
                idxs.append(e - 1)
            scene_samples.append(idxs)
            all_sample_idxs.extend(idxs)

        # Deduplicate so we don't run the net twice on scene-boundary frames.
        unique_idxs = sorted(set(all_sample_idxs))
        cache = {}
        for idx in tqdm(unique_idxs, desc='court sparse'):
            cache[idx] = self._infer_single_frame(frames[idx], scale_x, scale_y)

        for scene_i, ((s, e), idxs) in enumerate(zip(scenes, scene_samples)):
            sampled = [(i, *cache[i]) for i in idxs]
            valid = [(i, k, m) for (i, k, m) in sampled if k is not None]

            if len(valid) < min_samples_per_scene:
                print(f'[court-sparse] scene {scene_i} [{s}:{e}]: '
                      f'{len(valid)}/{len(idxs)} samples valid — marking all None '
                      f'(likely replay/tight/crowd)')
                continue

            stacked = np.stack([v[1].reshape(-1, 2) for v in valid])
            median_kps = np.median(stacked, axis=0)
            mad_xy = np.median(np.abs(stacked - median_kps), axis=0)
            max_jitter = float(np.linalg.norm(mad_xy, axis=1).max())

            if max_jitter < jitter_threshold_px:
                median_f = median_kps.reshape(-1, 1, 2).astype(np.float32)
                forward, _ = cv2.findHomography(refer_kps, median_f, method=0)
                if forward is None:
                    print(f'[court-sparse] scene {scene_i}: homography refit failed — leaving None')
                    continue
                inv_mat = cv2.invert(forward)[1]
                projected = cv2.perspectiveTransform(refer_kps, forward)
                print(f'[court-sparse] scene {scene_i} [{s}:{e}]: static '
                      f'(MAD={max_jitter:.2f}px, {len(valid)}/{len(idxs)} valid) '
                      f'— broadcast 1 H to {e-s} frames')
                for fi in range(s, e):
                    matrixes_res[fi] = inv_mat
                    kps_res[fi] = projected
            else:
                valid_arr = np.array([v[0] for v in valid])
                kps_by_idx = {v[0]: v[1] for v in valid}
                mat_by_idx = {v[0]: v[2] for v in valid}
                print(f'[court-sparse] scene {scene_i} [{s}:{e}]: moving '
                      f'(MAD={max_jitter:.2f}px) — nearest-sample fill over {e-s} frames')
                for fi in range(s, e):
                    nearest = int(valid_arr[np.argmin(np.abs(valid_arr - fi))])
                    matrixes_res[fi] = mat_by_idx[nearest]
                    kps_res[fi] = kps_by_idx[nearest]

        return matrixes_res, kps_res


In [ ]:
%%writefile /content/TennisProject/main.py
import cv2
from court_detection_net import CourtDetectorNet
import numpy as np
from court_reference import CourtReference
from bounce_detector import BounceDetector
from person_detector import PersonDetector
from ball_detector import BallDetector
from utils import scene_detect
import argparse
import torch


def smooth_player_tracks(persons_top, persons_bottom, max_jump_px=300, max_carry_frames=15):
    """Enforce temporal continuity on per-frame player picks.

    Rejects detections whose foot-point jumps > max_jump_px from the last accepted one
    (catches cases where filter_players picks up an umpire / ball kid / fence spectator
    when the actual player was missed by Faster R-CNN). Carries forward the last
    accepted bbox for up to max_carry_frames frames when the current frame has no
    valid candidate.
    """
    def _smooth(per_frame, label):
        out, last_box, last_foot, carry = [], None, None, 0
        rejected = carried = 0
        for frame_picks in per_frame:
            accepted = False
            if frame_picks:
                bbox, foot = frame_picks[0]
                if last_foot is None or np.linalg.norm(np.subtract(foot, last_foot)) <= max_jump_px:
                    out.append([(bbox, foot)])
                    last_box, last_foot, carry = bbox, foot, 0
                    accepted = True
                else:
                    rejected += 1
            if not accepted:
                if last_box is not None and carry < max_carry_frames:
                    out.append([(last_box, last_foot)])
                    carry += 1
                    carried += 1
                else:
                    out.append([])
                    last_box = last_foot = None
                    carry = 0
        print(f"[smooth] {label}: rejected {rejected} outlier(s), carried {carried} frame(s)")
        return out

    return _smooth(persons_top, "top"), _smooth(persons_bottom, "bottom")


def stabilize_static_camera(homography_matrices, kps_court, jitter_threshold_px=15.0):
    """Collapse per-frame court keypoints to a single median set if the camera is static.

    Detects static vs moving via per-keypoint MAD across frames. If max MAD < threshold,
    broadcasts the median keypoints and a single refit homography to every frame. Otherwise
    returns the inputs unchanged.
    """
    from homography import refer_kps
    valid = [i for i, k in enumerate(kps_court) if k is not None]
    if len(valid) < 5:
        return homography_matrices, kps_court

    stacked = np.stack([np.asarray(kps_court[i]).reshape(-1, 2) for i in valid])
    median_kps = np.median(stacked, axis=0)
    mad_xy = np.median(np.abs(stacked - median_kps), axis=0)
    max_jitter = float(np.linalg.norm(mad_xy, axis=1).max())

    if max_jitter > jitter_threshold_px:
        print(f"[stabilize] camera motion (max MAD={max_jitter:.2f}px > {jitter_threshold_px}px); keeping per-frame")
        return homography_matrices, kps_court

    median_f = median_kps.reshape(-1, 1, 2).astype(np.float32)
    forward, _ = cv2.findHomography(refer_kps, median_f, method=0)
    if forward is None:
        print("[stabilize] homography refit failed; keeping per-frame")
        return homography_matrices, kps_court

    inv_mat = cv2.invert(forward)[1]
    projected = cv2.perspectiveTransform(refer_kps, forward)
    N = len(kps_court)
    print(f"[stabilize] static camera (max MAD={max_jitter:.2f}px over {len(valid)}/{N} frames); broadcasting median")
    return [inv_mat] * N, [projected] * N

class _LazyFrames:
    """List-like lazy video-frame container.

    Upstream detectors consume `frames` by int index or slice (see ball_detector's
    3-frame sliding window). Eager-loading a full 1080p set (~4600 frames) needs
    ~28 GB RAM, which OOMs free-tier Colab (12 GB). This wrapper reads on demand
    and keeps a small LRU cache sized for ball_detector's access pattern.
    """
    def __init__(self, path, cache_size=8):
        self.path = path
        self.cap = cv2.VideoCapture(path)
        self.frame_count = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        self.fps_val = self.cap.get(cv2.CAP_PROP_FPS)
        self._cache = {}  # insertion-order LRU
        self._cache_size = cache_size
        self._next_pos = 0  # next frame idx that cap.read() will return

    def __len__(self):
        return self.frame_count

    def _read(self, i):
        if i in self._cache:
            # Move to LRU-end by re-inserting.
            f = self._cache.pop(i)
            self._cache[i] = f
            return f
        if i != self._next_pos:
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            self._next_pos = i
        ok, frame = self.cap.read()
        if not ok:
            raise IndexError(f"Could not read frame {i} from {self.path}")
        self._next_pos = i + 1
        self._cache[i] = frame
        if len(self._cache) > self._cache_size:
            oldest = next(iter(self._cache))
            del self._cache[oldest]
        return frame

    def __getitem__(self, key):
        if isinstance(key, slice):
            return [self._read(i) for i in range(*key.indices(self.frame_count))]
        if hasattr(key, '__index__'):  # covers int, np.integer
            i = int(key)
            if i < 0:
                i += self.frame_count
            if not 0 <= i < self.frame_count:
                raise IndexError(f"Frame index {i} out of range [0, {self.frame_count})")
            return self._read(i)
        raise TypeError(f"Unsupported frames index type: {type(key)}")

    def __iter__(self):
        for i in range(self.frame_count):
            yield self._read(i)

    def __del__(self):
        cap = getattr(self, 'cap', None)
        if cap is not None:
            cap.release()


def read_video(path_video):
    """Return (frames, fps) where frames is a list-like lazy reader.

    Preserves the upstream API (list-like random access + iteration) without
    loading every frame into RAM.
    """
    frames = _LazyFrames(path_video)
    return frames, int(frames.fps_val)

def get_court_img():
    court_reference = CourtReference()
    court = court_reference.build_court_reference()
    court = cv2.dilate(court, np.ones((10, 10), dtype=np.uint8))
    court_img = (np.stack((court, court, court), axis=2)*255).astype(np.uint8)
    return court_img

def main(frames, scenes, bounces, ball_track, homography_matrices, kps_court, persons_top, persons_bottom,
         draw_trace=False, trace=7):
    """
    :params
        frames: list of original images
        scenes: list of beginning and ending of video fragment
        bounces: list of image numbers where ball touches the ground
        ball_track: list of (x,y) ball coordinates
        homography_matrices: list of homography matrices
        kps_court: list of 14 key points of tennis court
        persons_top: list of person bboxes located in the top of tennis court
        persons_bottom: list of person bboxes located in the bottom of tennis court
        draw_trace: whether to draw ball trace
        trace: the length of ball trace
    :return
        imgs_res: list of resulting images
    """
    imgs_res = []
    width_minimap = 166
    height_minimap = 350
    is_track = [x is not None for x in homography_matrices]
    for num_scene in range(len(scenes)):
        sum_track = sum(is_track[scenes[num_scene][0]:scenes[num_scene][1]])
        len_track = scenes[num_scene][1] - scenes[num_scene][0]

        eps = 1e-15
        scene_rate = sum_track/(len_track+eps)
        if (scene_rate > 0.5):
            court_img = get_court_img()

            for i in range(scenes[num_scene][0], scenes[num_scene][1]):
                img_res = frames[i]
                inv_mat = homography_matrices[i]

                # draw ball trajectory
                if ball_track[i][0]:
                    if draw_trace:
                        for j in range(0, trace):
                            if i-j >= 0:
                                if ball_track[i-j][0]:
                                    draw_x = int(ball_track[i-j][0])
                                    draw_y = int(ball_track[i-j][1])
                                    img_res = cv2.circle(frames[i], (draw_x, draw_y),
                                    radius=3, color=(0, 255, 0), thickness=2)
                    else:
                        img_res = cv2.circle(img_res , (int(ball_track[i][0]), int(ball_track[i][1])), radius=5,
                                             color=(0, 255, 0), thickness=2)
                        img_res = cv2.putText(img_res, 'ball',
                              org=(int(ball_track[i][0]) + 8, int(ball_track[i][1]) + 8),
                              fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                              fontScale=0.8,
                              thickness=2,
                              color=(0, 255, 0))

                # draw court keypoints
                if kps_court[i] is not None:
                    for j in range(len(kps_court[i])):
                        img_res = cv2.circle(img_res, (int(kps_court[i][j][0, 0]), int(kps_court[i][j][0, 1])),
                                          radius=0, color=(0, 0, 255), thickness=10)

                height, width, _ = img_res.shape

                # draw bounce in minimap
                if i in bounces and inv_mat is not None:
                    ball_point = ball_track[i]
                    ball_point = np.array(ball_point, dtype=np.float32).reshape(1, 1, 2)
                    ball_point = cv2.perspectiveTransform(ball_point, inv_mat)
                    court_img = cv2.circle(court_img, (int(ball_point[0, 0, 0]), int(ball_point[0, 0, 1])),
                                                       radius=0, color=(0, 255, 255), thickness=50)

                minimap = court_img.copy()

                # draw persons
                persons = persons_top[i] + persons_bottom[i]
                for j, person in enumerate(persons):
                    if len(person[0]) > 0:
                        person_bbox = list(person[0])
                        img_res = cv2.rectangle(img_res, (int(person_bbox[0]), int(person_bbox[1])),
                                                (int(person_bbox[2]), int(person_bbox[3])), [255, 0, 0], 2)

                        # transmit person point to minimap
                        person_point = list(person[1])
                        person_point = np.array(person_point, dtype=np.float32).reshape(1, 1, 2)
                        person_point = cv2.perspectiveTransform(person_point, inv_mat)
                        minimap = cv2.circle(minimap, (int(person_point[0, 0, 0]), int(person_point[0, 0, 1])),
                                                           radius=0, color=(255, 0, 0), thickness=80)

                minimap = cv2.resize(minimap, (width_minimap, height_minimap))
                img_res[30:(30 + height_minimap), (width - 30 - width_minimap):(width - 30), :] = minimap
                imgs_res.append(img_res)

        else:
            imgs_res = imgs_res + frames[scenes[num_scene][0]:scenes[num_scene][1]]
    return imgs_res

def write(imgs_res, fps, path_output_video):
    height, width = imgs_res[0].shape[:2]
    out = cv2.VideoWriter(path_output_video, cv2.VideoWriter_fourcc(*'DIVX'), fps, (width, height))
    for num in range(len(imgs_res)):
        frame = imgs_res[num]
        out.write(frame)
    out.release()


if __name__ == '__main__':

    parser = argparse.ArgumentParser()
    parser.add_argument('--path_ball_track_model', type=str, help='path to pretrained model for ball detection')
    parser.add_argument('--path_court_model', type=str, help='path to pretrained model for court detection')
    parser.add_argument('--path_bounce_model', type=str, help='path to pretrained model for bounce detection')
    parser.add_argument('--path_input_video', type=str, help='path to input video')
    parser.add_argument('--path_output_video', type=str, default=None, help='path to output video (optional)')
    parser.add_argument('--path_upstream_artifact', type=str, default=None,
                        help='path to save upstream.npz (detection intermediates)')
    args = parser.parse_args()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    frames, fps = read_video(args.path_input_video)
    scenes = scene_detect(args.path_input_video)

    print('ball detection')
    ball_detector = BallDetector(args.path_ball_track_model, device)
    ball_track = ball_detector.infer_model(frames)

    print('court detection (scene-aware sparse)')
    court_detector = CourtDetectorNet(args.path_court_model, device)
    # Scene-aware sparse inference: ~1 sample/s per scene, then per-scene
    # stabilize (static → one H broadcast; moving → nearest-sample fill;
    # replay/tight → None → downstream skips). Replaces the per-frame dense
    # path plus the stabilize_static_camera pass that used to follow it.
    homography_matrices, kps_court = court_detector.infer_model_sparse(
        frames, scenes, fps=fps,
    )

    print('person detection')
    person_detector = PersonDetector(device)
    # Tier 1: request all candidates per side (filter_players=False). Court-polygon
    # filtering, temporal selection, and carry-forward all move to Phase 2 where they
    # operate on richer data and can be iterated without re-running Colab. Removing
    # smooth_player_tracks here for the same reason — Phase 2's tracker supersedes it.
    persons_top, persons_bottom = person_detector.track_players(frames, homography_matrices, filter_players=False)

    # bounce detection
    bounce_detector = BounceDetector(args.path_bounce_model)
    x_ball = [x[0] for x in ball_track]
    y_ball = [x[1] for x in ball_track]
    bounces = bounce_detector.predict(x_ball, y_ball)

    # Dump intermediates for downstream stages.
    if args.path_upstream_artifact:
        import hashlib
        import sys
        # Make src/tennis_vision importable regardless of where main.py is invoked from.
        import os
        _repo_src = os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '..', 'src')
        sys.path.insert(0, '/content/tennis-vision/src' if '/content' in args.path_input_video else _repo_src)
        from tennis_vision.io.artifacts import UpstreamArtifact, save_upstream

        _hasher = hashlib.sha256()
        with open(args.path_input_video, 'rb') as f:
            for _chunk in iter(lambda: f.read(1 << 20), b''):
                _hasher.update(_chunk)
        video_sha = _hasher.hexdigest()

        ball_track_arr = np.array(
            [[x if x is not None else np.nan, y if y is not None else np.nan] for x, y in ball_track],
            dtype=np.float32,
        )

        artifact = UpstreamArtifact(
            video_path=args.path_input_video,
            video_sha256=video_sha,
            fps=float(fps),
            frame_count=len(frames),
            ball_track=ball_track_arr,
            homography_matrices=homography_matrices,
            kps_court=kps_court,
            persons_top=persons_top,
            persons_bottom=persons_bottom,
            bounces=bounces,
            scenes=scenes,
        )
        save_upstream(artifact, args.path_upstream_artifact)
        print(f'[upstream] wrote {args.path_upstream_artifact}')

    if args.path_output_video:
        imgs_res = main(frames, scenes, bounces, ball_track, homography_matrices, kps_court,
                        persons_top, persons_bottom, draw_trace=True)
        write(imgs_res, fps, args.path_output_video)







In [ ]:
%%writefile /content/TennisProject/person_detector.py
import cv2
import torch
from court_reference import CourtReference
from scipy import signal
import numpy as np
from scipy.spatial import distance
from tqdm import tqdm
from ultralytics import YOLO  # type: ignore[import-not-found]


class PersonDetector():
    """Tier 2 swap: Faster R-CNN ResNet50 FPN → YOLOv8 for far-player recall.

    YOLOv8x raises AP@small on COCO another ~5 pp over -m, which is the
    difference that matters for the broadcast top player at ~60-100 px tall
    (where the smaller -m variant misses entire frames). Interface kept
    identical (detect/track_players/filter_players) so the surrounding
    upstream code is unchanged.
    """
    def __init__(self, device='cpu', model_name='yolov8x.pt'):
        self.detection_model = YOLO(model_name)
        # Warm-up forward pass — pre-loads weights so progress bars aren't delayed.
        self.device = device
        self.court_ref = CourtReference()
        self.ref_top_court = self.court_ref.get_court_mask(2)
        self.ref_bottom_court = self.court_ref.get_court_mask(1)
        self.point_person_top = None
        self.point_person_bottom = None
        self.counter_top = 0
        self.counter_bottom = 0

    def detect(self, image, person_min_score=0.3):
        # COCO class 0 = 'person'. YOLOv8 accepts BGR numpy directly; no need
        # to manually tensorize. imgsz defaults to 640 — appropriate for the
        # model's training resolution. For 1920×1080 input this means YOLOv8
        # letterboxes down to 640 internally, returning boxes in original coords.
        PERSON_CLASS = 0
        results = self.detection_model(
            image,
            conf=person_min_score,
            classes=[PERSON_CLASS],
            device=self.device,
            verbose=False,
        )
        persons_boxes = []
        probs = []
        if results and results[0].boxes is not None:
            for box, conf in zip(results[0].boxes.xyxy, results[0].boxes.conf):
                persons_boxes.append(box.cpu().numpy())
                probs.append(float(conf.cpu()))
        return persons_boxes, probs

    def detect_top_and_bottom_players(self, image, inv_matrix, filter_players=False):
        matrix = cv2.invert(inv_matrix)[1]
        mask_top_court = cv2.warpPerspective(self.ref_top_court, matrix, image.shape[1::-1])
        mask_bottom_court = cv2.warpPerspective(self.ref_bottom_court, matrix, image.shape[1::-1])
        person_bboxes_top, person_bboxes_bottom = [], []

        bboxes, probs = self.detect(image, person_min_score=0.3)
        if len(bboxes) > 0:
            person_points = [[int((bbox[2] + bbox[0]) / 2), int(bbox[3])] for bbox in bboxes]
            person_bboxes = list(zip(bboxes, person_points))

            person_bboxes_top = [pt for pt in person_bboxes if mask_top_court[pt[1][1]-1, pt[1][0]] == 1]
            person_bboxes_bottom = [pt for pt in person_bboxes if mask_bottom_court[pt[1][1] - 1, pt[1][0]] == 1]

            if filter_players:
                person_bboxes_top, person_bboxes_bottom = self.filter_players(person_bboxes_top, person_bboxes_bottom,
                                                                              matrix)
        return person_bboxes_top, person_bboxes_bottom

    def filter_players(self, person_bboxes_top, person_bboxes_bottom, matrix):
        """
        Leave one person at the top and bottom of the tennis court
        """
        refer_kps = np.array(self.court_ref.key_points[12:], dtype=np.float32).reshape((-1, 1, 2))
        trans_kps = cv2.perspectiveTransform(refer_kps, matrix)
        center_top_court = trans_kps[0][0]
        center_bottom_court = trans_kps[1][0]
        if len(person_bboxes_top) > 1:
            dists = [distance.euclidean(x[1], center_top_court) for x in person_bboxes_top]
            ind = dists.index(min(dists))
            person_bboxes_top = [person_bboxes_top[ind]]
        if len(person_bboxes_bottom) > 1:
            dists = [distance.euclidean(x[1], center_bottom_court) for x in person_bboxes_bottom]
            ind = dists.index(min(dists))
            person_bboxes_bottom = [person_bboxes_bottom[ind]]
        return person_bboxes_top, person_bboxes_bottom

    def track_players(self, frames, matrix_all, filter_players=False):
        persons_top = []
        persons_bottom = []
        min_len = min(len(frames), len(matrix_all))
        for num_frame in tqdm(range(min_len)):
            img = frames[num_frame]
            if matrix_all[num_frame] is not None:
                inv_matrix = matrix_all[num_frame]
                person_top, person_bottom = self.detect_top_and_bottom_players(img, inv_matrix, filter_players)
            else:
                person_top, person_bottom = [], []
            persons_top.append(person_top)
            persons_bottom.append(person_bottom)
        return persons_top, persons_bottom




In [ ]:
!mkdir -p /content/tennis-vision/src/tennis_vision/io

In [ ]:
%%writefile /content/tennis-vision/src/tennis_vision/__init__.py
"""Tennis match analysis pipeline."""

__version__ = "0.1.0"


In [ ]:
%%writefile /content/tennis-vision/src/tennis_vision/io/__init__.py
"""Video I/O, hashing, artifact schemas."""


In [ ]:
%%writefile /content/tennis-vision/src/tennis_vision/io/artifacts.py
"""Load/save intermediate artifacts produced by the yastrebksv pipeline.

NOTE: `load_upstream` uses `np.load(..., allow_pickle=True)` because the ragged
object arrays (per-frame person picks, per-frame homographies that may be None)
cannot be stored as regular numpy arrays. This is safe because we only ever
load files we produced ourselves; never load an upstream.npz from an untrusted
source.
"""
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import numpy as np


PersonPick = tuple[np.ndarray, list[int]]  # (bbox [x1,y1,x2,y2], foot [x, y])


@dataclass
class UpstreamArtifact:
    video_path: str
    video_sha256: str
    fps: float
    frame_count: int
    ball_track: np.ndarray                        # (N, 2) float32, NaN where missing
    homography_matrices: list[np.ndarray | None]  # N items, 3x3 float32 or None
    kps_court: list[np.ndarray | None]            # N items, (14, 1, 2) float32 or None
    persons_top: list[list[PersonPick]]           # N items
    persons_bottom: list[list[PersonPick]]        # N items
    bounces: set[int]
    scenes: list[tuple[int, int]]


def save_upstream(artifact: UpstreamArtifact, path: Path | str) -> None:
    """Save artifact to a single .npz file. Object arrays are used for ragged fields."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    homography_arr = np.empty(artifact.frame_count, dtype=object)
    for i, h in enumerate(artifact.homography_matrices):
        homography_arr[i] = h  # None preserved as None

    kps_arr = np.empty(artifact.frame_count, dtype=object)
    for i, k in enumerate(artifact.kps_court):
        kps_arr[i] = k

    top_arr = np.empty(artifact.frame_count, dtype=object)
    bot_arr = np.empty(artifact.frame_count, dtype=object)
    for i in range(artifact.frame_count):
        top_arr[i] = artifact.persons_top[i]
        bot_arr[i] = artifact.persons_bottom[i]

    scenes_data = np.array(artifact.scenes, dtype=np.int32)
    if scenes_data.ndim == 1 and len(scenes_data) == 0:
        scenes_data = scenes_data.reshape(0, 2)

    np.savez_compressed(
        path,
        video_path=np.array(artifact.video_path),
        video_sha256=np.array(artifact.video_sha256),
        fps=np.array(artifact.fps),
        frame_count=np.array(artifact.frame_count),
        ball_track=artifact.ball_track,
        homography_matrices=homography_arr,
        kps_court=kps_arr,
        persons_top=top_arr,
        persons_bottom=bot_arr,
        bounces=np.array(sorted(artifact.bounces), dtype=np.int32),
        scenes=scenes_data,
    )


def load_upstream(path: Path | str) -> UpstreamArtifact:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Upstream artifact not found: {path}")
    with np.load(path, allow_pickle=True) as data:
        return UpstreamArtifact(
            video_path=str(data["video_path"]),
            video_sha256=str(data["video_sha256"]),
            fps=float(data["fps"]),
            frame_count=int(data["frame_count"]),
            ball_track=data["ball_track"].astype(np.float32),
            homography_matrices=list(data["homography_matrices"]),
            kps_court=list(data["kps_court"]),
            persons_top=list(data["persons_top"]),
            persons_bottom=list(data["persons_bottom"]),
            bounces=set(int(x) for x in data["bounces"]),
            scenes=[(int(a), int(b)) for a, b in data["scenes"].reshape(-1, 2)],
        )


In [ ]:
!mkdir -p /content/TennisProject/weights
!gdown --quiet 1XEYZ4myUN7QT-NeBYJI0xteLsvs-ZAOl -O /content/TennisProject/weights/model_best.pt
!gdown --quiet 1f-Co64ehgq4uddcQm1aFBDtbnyZhQvgG -O /content/TennisProject/weights/model_tennis_court_det.pt
!gdown --quiet 1Eo5HDnAQE8y_FbOftKZ8pjiojwuy2BmJ -O /content/TennisProject/weights/ctb_regr_bounce.cbm
!ls -lh /content/TennisProject/weights/

### Upload input video

Pick your local `point1.mp4` or `set1.mp4`. For large files or repeated runs, prefer mounting Drive instead:
```python
from google.colab import drive
drive.mount('/content/drive')
input_video = '/content/drive/MyDrive/tennis-tracker/tennisVideo.mp4'
```

In [ ]:
# Drive-mount path: set1.mp4 at the top level of My Drive.
from google.colab import drive
drive.mount('/content/drive')
input_video = '/content/drive/MyDrive/tennis-tracker/tennisVideo.mp4'
print('Input:', input_video)

# Alternative: browser upload (slow for large files; prefer Drive).
# from google.colab import files
# uploaded = files.upload()
# input_video = '/content/' + next(iter(uploaded.keys()))
# print('Input:', input_video)

In [ ]:
import os, time
os.makedirs('/content/outputs', exist_ok=True)
upstream_path = '/content/outputs/upstream.npz'
# Upstream's overlay render materializes frame slices into RAM — fine for a
# short point but OOMs on a full set. Skip it here; render the richer Phase 2
# overlay locally from the downloaded upstream.npz via `tennis-vision render`.
t0 = time.time()
!cd /content/TennisProject && python main.py \
    --path_ball_track_model weights/model_best.pt \
    --path_court_model weights/model_tennis_court_det.pt \
    --path_bounce_model weights/ctb_regr_bounce.cbm \
    --path_input_video "$input_video" \
    --path_upstream_artifact "$upstream_path"
print(f'Total wall time: {time.time()-t0:.1f}s')

In [ ]:
from google.colab import files
files.download('/content/outputs/upstream.npz')